In [ ]:
GPT_CONFIG_124M={
    "vocab_size":50257,
    "context_length":1024,
    "emb_dim":768,
    "n_heads":12,
    "n_layers":12,
    "drop_rate":0.1,
    "qkv_bias":False
}

In [ ]:
import torch
import torch.nn as nn

class DummyGPTMOdel(nn.Module):
  def __init__(self,cfg):
    super().__init__()
    self.tok_emb=nn.Embedding(cfg["vocab_size"],cfg["emb_dim"])
    self.pos_emb=nn.Embedding(cfg["context_length"],cfg["emb_dim"])
    self.drop_emb=nn.Dropout(cfg["drop_rate"])

    self.trf_blocks=nn.Sequential(
        *[DummyTransformerBlock(cfg) for _ in range(cfg["n_layers"])]
    )

    self.final_norm=DummyLayerNorm(cfg["emb_dim"])
    self.out_head=nn.Linear(cfg["emb_dim"],cfg["vocab_size"],bias=False)

    def forward(self,in_idx):
      batch_size,seq_len=in_idx.shape
      tok_embeds=self.tok_emb(in_idx)
      pos_embeds=self.pos_emb(torch.arange(seq_len,device=in_idx.device))
      x=tok_embeds+pos_embeds
      x=self.drop_emb(x)
      x=self.trf_blocks(x)
      x=self.final_norm(x)
      logits=self.out_head(x)
      return logits
class DummyTransformerBlock(nn.Module):
  def __init__(self,cfg):
    super().__init__()
  def forward(self,x):
    return x
class DummyLayerNorm(nn.Module):
  def __init__(self,cfg):
    super().__init__()
  def forward(self,x):
    return x

In [ ]:
import torch
import torch.nn as nn

class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

In [ ]:
torch.manual_seed(1337)
batch=torch.randn(2,5)
layer=nn.Sequential(nn.Linear(5,6),nn.ReLU())
ln = LayerNorm(5)
res=ln(batch)
print(res.mean(dim=1),res.var(dim=1, unbiased=False))

tensor([2.3842e-07, 5.9605e-09], grad_fn=<MeanBackward1>) tensor([1.0000, 0.9999], grad_fn=<VarBackward0>)


In [ ]:
class GELU(nn.Module):
  def __init__(self):
    super().__init__()
  def forward(self,x):
    return 0.5*x*(1+torch.tanh(torch.sqrt(torch.tensor(2/torch.pi))*(x+0.044715*torch.pow(x,3))))

In [ ]:
class FeedForward(nn.Module):
  def __init__(self,cfg):
    super().__init__()
    self.layers=nn.Sequential(
        nn.Linear(cfg["emb_dim"],4*cfg["emb_dim"]),
        GELU(),
        nn.Linear(4*cfg["emb_dim"],cfg["emb_dim"]),
    )
  def forward(self,x):
    return self.layers(x)

In [ ]:
ffn=FeedForward(GPT_CONFIG_124M)
x=torch.rand(2,3,768)
print(ffn(x))

tensor([[[-0.2058,  0.0393,  0.1041,  ...,  0.1716, -0.0190,  0.1892],
         [-0.0675,  0.0570,  0.1484,  ...,  0.1153,  0.0977,  0.2683],
         [-0.1145, -0.0583,  0.0568,  ...,  0.1770,  0.0896,  0.1495]],

        [[-0.1161,  0.0355,  0.1438,  ...,  0.0391,  0.0188,  0.1313],
         [-0.0572,  0.1086,  0.0916,  ...,  0.0675,  0.0370,  0.0831],
         [-0.1064,  0.0708,  0.0158,  ...,  0.1839,  0.1261,  0.2011]]],
       grad_fn=<ViewBackward0>)


In [ ]:
class ExampleDeepNeuralNetwork(nn.Module):
  def __init__(self,layers_sizes,use_shortcut):
    super().__init__()
    self.use_shortcut=use_shortcut
    self.layers=nn.ModuleList([
        nn.Sequential(nn.Linear(layers_sizes[0],layers_sizes[1]),GELU()),
        nn.Sequential(nn.Linear(layers_sizes[1],layers_sizes[2]),GELU()),
        nn.Sequential(nn.Linear(layers_sizes[2],layers_sizes[3]),GELU()),
        nn.Sequential(nn.Linear(layers_sizes[3],layers_sizes[4]),GELU()),
        nn.Sequential(nn.Linear(layers_sizes[4],layers_sizes[5]),GELU())
    ])
  def forward(self,x):
    for layer in self.layers:
      layer_output=layer(x)
      if self.use_shortcut and x.shape==layer_output.shape:
        x=x+layer_output
      else:
        x=layer_output
    return x


In [ ]:
layer_size=[3,3,3,3,3,1]
sample_input=torch.tensor([[1.,0.,-1.]])
torch.manual_seed(123)
model_without_shortcut=ExampleDeepNeuralNetwork(layer_size,False)
model_with_shortcut=ExampleDeepNeuralNetwork(layer_size,True)

In [ ]:
def print_gradients(model, x):
    model.zero_grad()  # important: clear old gradients

    output = model(x)

    target = torch.zeros_like(output)

    loss_fn = nn.MSELoss()
    loss = loss_fn(output, target)

    loss.backward()

    for name, param in model.named_parameters():
        if 'weight' in name:
            print(f"{name} has gradient mean of {param.grad.abs().mean().item()}")

In [ ]:
print_gradients(model_without_shortcut,sample_input)
print()
print_gradients(model_with_shortcut,sample_input)

layers.0.0.weight has gradient mean of 0.00020173584925942123
layers.1.0.weight has gradient mean of 0.00012011159560643137
layers.2.0.weight has gradient mean of 0.0007152040489017963
layers.3.0.weight has gradient mean of 0.0013988736318424344
layers.4.0.weight has gradient mean of 0.005049645435065031

layers.0.0.weight has gradient mean of 0.0014432291500270367
layers.1.0.weight has gradient mean of 0.004846952389925718
layers.2.0.weight has gradient mean of 0.004138893447816372
layers.3.0.weight has gradient mean of 0.005915115587413311
layers.4.0.weight has gradient mean of 0.032659437507390976


In [ ]:
class MultiHeadAttention(nn.Module):
  def __init__(self,d_in,d_out,context_length,dropout,n_heads,qkv_bias=False):
    super().__init__()
    assert(d_out % n_heads==0),\
      "d_out must be divisible by num_heads"
    self.d_out=d_out
    self.num_heads=n_heads
    self.head_dim=d_out//n_heads

    self.W_query=nn.Linear(d_in,d_out,bias=qkv_bias)
    self.W_key=nn.Linear(d_in,d_out,bias=qkv_bias)
    self.W_value=nn.Linear(d_in,d_out,bias=qkv_bias)
    self.out_proj=nn.Linear(d_out,d_out)
    self.dropout=nn.Dropout(dropout)
    self.register_buffer("mask",torch.triu(torch.ones(context_length,context_length),diagonal=1))


  def forward(self,x):
    b,num_tokens,d_in=x.shape

    keys=self.W_key(x)
    queries=self.W_query(x)
    values=self.W_value(x)

    keys=keys.view(b,num_tokens,self.num_heads,self.head_dim)
    queries=queries.view(b,num_tokens,self.num_heads,self.head_dim)
    values=values.view(b,num_tokens,self.num_heads,self.head_dim)

    keys=keys.transpose(1,2)
    queries=queries.transpose(1,2)
    values=values.transpose(1,2)

    attn_scores=queries @ keys.transpose(2,3)
    attn_scores.masked_fill_(
        self.mask.bool()[:num_tokens,:num_tokens],-torch.inf)
    attn_weights=torch.softmax(attn_scores/keys.shape[-1]**0.5,dim=-1)
    attn_weights=self.dropout(attn_weights)
    context_vec=(attn_weights @ values).transpose(1,2)
    context_vec=context_vec.contiguous().view(b,num_tokens,self.d_out)
    context_vec=self.out_proj(context_vec)
    return context_vec


In [ ]:
class TransformerBlock(nn.Module):
  def __init__(self,cfg):
    super().__init__()
    self.att=MultiHeadAttention(
        d_in=cfg["emb_dim"],
        d_out=cfg["emb_dim"],
        context_length=cfg["context_length"],
        n_heads=cfg["n_heads"],
        dropout=cfg["drop_rate"],
        qkv_bias=cfg["qkv_bias"]
    )
    self.ff=FeedForward(cfg)
    self.norm1=LayerNorm(cfg["emb_dim"])
    self.norm2=LayerNorm(cfg["emb_dim"])
    self.drop_shortcut=nn.Dropout(cfg["drop_rate"])
  def forward(self,x):
    shortcut=x
    x=self.norm1(x)
    x=self.att(x)
    x=self.drop_shortcut(x)
    x=x+shortcut

    shortcut=x
    x=self.norm2(x)
    x=self.ff(x)
    x=self.drop_shortcut(x)
    x=x+shortcut
    return x

In [ ]:
torch.manual_seed(123)
x=torch.rand(2,4,768)
block=TransformerBlock(GPT_CONFIG_124M)
op=block(x)
print(x.shape)
print(op.shape)

torch.Size([2, 4, 768])
torch.Size([2, 4, 768])


In [ ]:
import torch
import torch.nn as nn


# ------------------ GELU ------------------

class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (
            1 + torch.tanh(
                torch.sqrt(torch.tensor(2 / torch.pi, device=x.device)) *
                (x + 0.044715 * torch.pow(x, 3))
            )
        )


# ------------------ Feed Forward ------------------

class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]), ## Expansion
            GELU(), ## Activation
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]), ## Contraction
        )

    def forward(self, x):
        return self.layers(x)


# ------------------ MultiHead Attention ------------------

class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, n_heads, qkv_bias=False):
        super().__init__()

        assert (d_out % n_heads == 0)

        self.d_out = d_out
        self.num_heads = n_heads
        self.head_dim = d_out // n_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)

        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)

        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)

        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf
        )

        attn_weights = torch.softmax(
            attn_scores / (self.head_dim ** 0.5),
            dim=-1
        )

        attn_weights = self.dropout(attn_weights)

        context_vec = (attn_weights @ values).transpose(1, 2)
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)

        return context_vec


# ------------------ LayerNorm ------------------

class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift


# ------------------ Transformer Block ------------------

class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            n_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"]
        )

        self.ff = FeedForward(cfg)

        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])

        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        return x


# ------------------ GPT Model ------------------

class GPTMOdel(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape

        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))

        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)

        x = self.trf_blocks(x)
        x = self.final_norm(x)

        logits = self.out_head(x)

        return logits

In [ ]:
import torch

torch.manual_seed(123)

x = torch.randint(0, GPT_CONFIG_124M["vocab_size"], (2, 4))

gpt = GPTMOdel(GPT_CONFIG_124M)

op = gpt(x)

print("Input shape:", x.shape)
print("Output shape:", op.shape)

Input shape: torch.Size([2, 4])
Output shape: torch.Size([2, 4, 50257])


In [ ]:
tp=sum(p.numel() for p in gpt.parameters())
print(f"Total parameters: {tp}")

Total parameters: 163009536


In [ ]:
tp1=tp-sum(p.numel() for p in gpt.out_head.parameters())
print("total parameters without out_head",tp1)

total parameters without out_head 124412160


In [ ]:
total_size=tp*4
total_size_mb=total_size/pow(1024,2)
print(f"total size of the model: {total_size_mb:.2f} MB")

total size of the model: 621.83 MB


In [ ]:
def generate_text_simple(model, idx, max_new_tokens,context_size):
  for _ in range(max_new_tokens):
    idx_cond=idx[:,-context_size:]
    with torch.no_grad():
      logits=model(idx_cond)
    logits=logits[:,-1,:]
    probas=torch.softmax(logits,dim=-1)
    idx_next=torch.argmax(probas,dim=1,keepdim=True)
    idx=torch.cat((idx,idx_next),dim=1)
    print(idx)
  return idx

In [ ]:
import tiktoken
tokenizer=tiktoken.get_encoding("gpt2")
s="Hello, I am"
enc=tokenizer.encode(s)
print(enc)
encoded_tensor=torch.tensor(enc).unsqueeze(0)
print(encoded_tensor.shape)

[15496, 11, 314, 716]
torch.Size([1, 4])


In [ ]:
gpt.eval()
out=generate_text_simple(
    model=gpt,
    idx=encoded_tensor,
    max_new_tokens=6,
    context_size=GPT_CONFIG_124M["context_length"]
)
print(out)
print(len(out[0]))

tensor([[15496,    11,   314,   716, 49562]])
tensor([[15496,    11,   314,   716, 49562, 16024]])
tensor([[15496,    11,   314,   716, 49562, 16024,  4909]])
tensor([[15496,    11,   314,   716, 49562, 16024,  4909,  9667]])
tensor([[15496,    11,   314,   716, 49562, 16024,  4909,  9667, 38598]])
tensor([[15496,    11,   314,   716, 49562, 16024,  4909,  9667, 38598,  3258]])
tensor([[15496,    11,   314,   716, 49562, 16024,  4909,  9667, 38598,  3258]])
10


In [ ]:
dc=tokenizer.decode(out[0].tolist())
print(dc)

Hello, I amFra Bio contains dates creepingarr


In [ ]:
GPT_CONFIG_124M={
    "vocab_size":50257,
    "context_length":256,
    "emb_dim":768,
    "n_heads":12,
    "n_layers":12,
    "drop_rate":0.1,
    "qkv_bias":False
}
torch.manual_seed(123)
model=GPTMOdel(GPT_CONFIG_124M)
model.eval();

In [ ]:
import tiktoken

def text_to_token(text,tokenizer):
  encoded=tokenizer.encode(text,allowed_special={'<|endoftext|>'})
  encoded_tensor=torch.tensor(encoded).unsqueeze(0)
  return encoded_tensor
def token_ids_to_text(token_ids,tokenizer):
  flat=token_ids.squeeze(0)
  return tokenizer.decode(flat.tolist())
start_context="Every effot moves you"
tokenizer=tiktoken.get_encoding("gpt2")
token_ids=generate_text_simple(
    model=model,
    idx=text_to_token(start_context,tokenizer),
    max_new_tokens=10,
    context_size=GPT_CONFIG_124M["context_length"]
)
print(token_ids_to_text(token_ids,tokenizer))

tensor([[ 6109,   914,   313,  6100,   345, 30057]])
tensor([[ 6109,   914,   313,  6100,   345, 30057,  6287]])
tensor([[ 6109,   914,   313,  6100,   345, 30057,  6287, 45750]])
tensor([[ 6109,   914,   313,  6100,   345, 30057,  6287, 45750, 23879]])
tensor([[ 6109,   914,   313,  6100,   345, 30057,  6287, 45750, 23879, 24071]])
tensor([[ 6109,   914,   313,  6100,   345, 30057,  6287, 45750, 23879, 24071,
          7025]])
tensor([[ 6109,   914,   313,  6100,   345, 30057,  6287, 45750, 23879, 24071,
          7025, 27362]])
tensor([[ 6109,   914,   313,  6100,   345, 30057,  6287, 45750, 23879, 24071,
          7025, 27362, 48328]])
tensor([[ 6109,   914,   313,  6100,   345, 30057,  6287, 45750, 23879, 24071,
          7025, 27362, 48328, 39654]])
tensor([[ 6109,   914,   313,  6100,   345, 30057,  6287, 45750, 23879, 24071,
          7025, 27362, 48328, 39654, 49641]])
Every effot moves you261 extentNarr electorateordered broadcastacting aboriginal Pesh763


In [ ]:
input=torch.tensor([[16833,3626,6100], #every effort moves
                    [40,1107,588]])    #I really like
targets=torch.tensor([[3626,6100,345], #effort moves you
                      [1107,588,11311]]) #really like chocolate

In [ ]:
with torch.no_grad():
  logits=model(input)

probas=torch.softmax(logits,dim=-1)
print(probas)
print(logits.shape)

tensor([[[1.8849e-05, 1.5172e-05, 1.1687e-05,  ..., 2.2409e-05,
          6.9776e-06, 1.8776e-05],
         [9.1569e-06, 1.0062e-05, 7.8786e-06,  ..., 2.9090e-05,
          6.0103e-06, 1.3571e-05],
         [2.9877e-05, 8.8507e-06, 1.5741e-05,  ..., 3.5456e-05,
          1.4094e-05, 1.3526e-05]],

        [[1.2561e-05, 2.0538e-05, 1.4332e-05,  ..., 1.0389e-05,
          3.4784e-05, 1.4239e-05],
         [7.2731e-06, 1.7864e-05, 1.0565e-05,  ..., 2.1206e-05,
          1.1390e-05, 1.5559e-05],
         [2.9496e-05, 3.3605e-05, 4.1029e-05,  ..., 6.5249e-06,
          5.8203e-05, 1.3698e-05]]])
torch.Size([2, 3, 50257])


In [ ]:
token_ids=torch.argmax(probas,dim=-1,keepdim=True)
print(token_ids)

tensor([[[16657],
         [  339],
         [42826]],

        [[49906],
         [29669],
         [41751]]])


In [ ]:
print(token_ids_to_text(token_ids[0].flatten(),tokenizer))

 Armed heNetflix


In [ ]:
target_probas_1=probas[0,[0,1,2],targets[0]]
target_probas_2=probas[1,[0,1,2],targets[1]]
print(target_probas_1)
print(target_probas_2)

tensor([7.4540e-05, 3.1061e-05, 1.1563e-05])
tensor([1.0337e-05, 5.6776e-05, 4.7559e-06])


In [ ]:
logits=logits.flatten(0,1)
targets=targets.flatten()
print(logits.shape)
print(targets.shape)

torch.Size([6, 50257])
torch.Size([6])


In [ ]:
loss=torch.nn.functional.cross_entropy(logits,targets)
print(loss)

tensor(10.7940)


In [ ]:
perplexity=torch.exp(loss)
print(perplexity)

tensor(48725.8203)


In [ ]:
with open("the_verdict.txt","r",encoding="utf-8") as file:
  text_data=file.read()
print(text_data[:99])

I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [ ]:
print(len(text_data))

20479


In [ ]:
total_tokens=len(tokenizer.encode(text_data))
print(total_tokens)

5145


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import tiktoken

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.output_ids = []

        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1:i + 1 + max_length]

            self.input_ids.append(torch.tensor(input_chunk))
            self.output_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.output_ids[idx]


def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128,
                         shuffle=True, drop_last=True, num_workers=0):

    tokenizer = tiktoken.get_encoding("gpt2")

    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [ ]:
train_ratio=0.90
split_idx=int(train_ratio*len(text_data))
train_data=text_data[:split_idx]
print(len(tokenizer.encode(train_data)))
val_data=text_data[split_idx:]
torch.manual_seed(123)
train_loader=create_dataloader_v1(
    train_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    shuffle=True,
    drop_last=True,
    num_workers=0
)
val_loader=create_dataloader_v1(
    val_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    shuffle=True,
    drop_last=True,
    num_workers=0
)

4612


In [ ]:
for x,y in train_loader:
  print(x.shape,y.shape)

torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])


In [ ]:
def cal_loss_batch(input_batch,target_batch,model,device):
  input_batch,target_batch=input_batch.to(device),target_batch.to(device)
  logits=model(input_batch)
  loss=torch.nn.functional.cross_entropy(logits.flatten(0,1),target_batch.flatten())
  return loss
def cal_loss_loader(data_loader,model,device,num_batches=None):
  total_loss=0
  num_batches=len(data_loader) if num_batches is None else min(num_batches,len(data_loader))
  for i,(input_batch,target_batch) in enumerate(data_loader):
    if i<num_batches:
      loss=cal_loss_batch(input_batch,target_batch,model,device)
      total_loss+=loss.item()
    else:
      break
  return total_loss/num_batches


In [ ]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
torch.manual_seed(123)
with torch.no_grad():
  train_loss=cal_loss_loader(train_loader,model,device)
  val_loss=cal_loss_loader(val_loader,model,device)
print(train_loss)
print(val_loss)

10.98758347829183
10.981106758117676


In [ ]:
def train_model_simple(model,train_loader,val_loader,optimizer,device,num_epochs,
                       eval_freq,eval_iter,start_context,tokenizer):

  train_losses,val_losses,track_tokens_seen=[],[],[]
  tokens_seen,global_step=0,-1

  for epoch in range(num_epochs):
    model.train()

    for input_batch,target_batch in train_loader:
      optimizer.zero_grad()
      loss=cal_loss_batch(input_batch,target_batch,model,device)
      loss.backward()
      optimizer.step()
      tokens_seen+=input_batch.numel()
      global_step+=1

      if global_step % eval_freq==0:
        train_loss,val_loss=evaluate_model(
            model,train_loader,val_loader,device,eval_iter
        )
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        track_tokens_seen.append(tokens_seen)
        print(f"Ep {epoch+1} (step {global_step:06d}):"
        f"Train loss {train_loss:.4f}, Val loss {val_loss:.4f}")
    generate_and_print_sample(
      model, tokenizer, device, start_context
    )
  return train_losses,val_losses,track_tokens_seen


In [ ]:
def evaluate_model(model,train_loader,val_loader,device,eval_iter):
  model.eval()
  with torch.no_grad():
    train_loss=cal_loss_loader(train_loader,model,device,num_batches=eval_iter)
    val_loss=cal_loss_loader(val_loader,model,device,num_batches=eval_iter)
  model.train()
  return train_loss,val_loss


In [ ]:
def generate_and_print_sample(model,tokenizer,device,start_context):
  model.eval()
  context_size=model.pos_emb.weight.shape[0]
  encoded=text_to_token(start_context,tokenizer).to(device)
  with torch.no_grad():
    token_ids=generate_text_simple(
        model=model,idx=encoded,
        max_new_tokens=50,context_size=context_size
    )
  decoded_text = token_ids_to_text(token_ids, tokenizer)
  print(decoded_text.replace("\n", " "))
  model.train()

In [ ]:
def generate_text_simple(model, idx, max_new_tokens,context_size):
  for _ in range(max_new_tokens):
    idx_cond=idx[:,-context_size:]
    with torch.no_grad():
      logits=model(idx_cond)
    logits=logits[:,-1,:]
    probas=torch.softmax(logits,dim=-1)
    idx_next=torch.argmax(probas,dim=1,keepdim=True)
    idx=torch.cat((idx,idx_next),dim=1)
  return idx

#Controling randomness

In [ ]:
model.to("cpu")
model.eval()

GPTMOdel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(256, 768)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=False)
        (W_key): Linear(in_features=768, out_features=768, bias=False)
        (W_value): Linear(in_features=768, out_features=768, bias=False)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features

In [ ]:
tokenizer=tiktoken.get_encoding("gpt2")
token_ids=generate_text_simple(
    model=model,
    idx=text_to_token("Every effort moves you",tokenizer),
    max_new_tokens=25,
    context_size=GPT_CONFIG_124M["context_length"]
)
print(token_ids_to_text(token_ids,tokenizer))

Every effort moves you rentingetic wasnم refres RexMeCHicular stren Mortgage TT remember gard ACTIONSussedOND Land Engeleddedemate breaths proxies GalaxyForm


#**Decoding** strategies

#Temperature Scaling
more temp value flattens distribution less temp value sharpens it

In [ ]:
#after applying softamx on logits probas instead of applying argmax apply multinomial
probas=torch.tensor([0.1,0.4,0.4,0.1])
next_token_id=torch.multinomial(probas,num_samples=1).item()
print(next_token_id)

1


In [ ]:
logits=torch.tensor([0.1,0.2,0.7,0.1])
a,b,c=0.1,2,5
logits1=logits/a
logits2=logits/b
logits3=logits/c
probas1=torch.softmax(logits1,dim=-1)
print(probas1)
probas2=torch.softmax(logits2,dim=-1)
print(probas2)
probas3=torch.softmax(logits3,dim=-1)
print(probas3)

tensor([0.0025, 0.0067, 0.9884, 0.0025])
tensor([0.2272, 0.2389, 0.3067, 0.2272])
tensor([0.2411, 0.2460, 0.2718, 0.2411])


#Top-k Sampling

In [ ]:
logits=torch.tensor([0.1,0.2,0.7,0.1])
tok_k=2
top_logits,top_ps=torch.topk(logits,k=tok_k)
print(top_logits)
print(top_ps)

tensor([0.7000, 0.2000])
tensor([2, 1])


In [ ]:
new_l=torch.where(
    condition=logits<top_logits[-1],
    input=torch.tensor(float("-inf")),
    other=logits
)
print(new_l)

tensor([  -inf, 0.2000, 0.7000,   -inf])


In [ ]:
print(torch.softmax(new_l,dim=0))

tensor([0.0000, 0.3775, 0.6225, 0.0000])


#merging temp and sample
logits->topk->divide by temp->softmax->multinomial

In [ ]:
def generate(model,idx,max_new_tokens,context_size,temp=0.0,top_k=None,eos_id=None):
  for _ in range(max_new_tokens):
    idx_cond=idx[:,-context_size:]
    with torch.no_grad():
      logits=model(idx_cond)
    logits=logits[:,-1,:]
    if top_k is not None:
      t_logits,_=torch.topk(logits,k=top_k)
      mini_val=t_logits[:,-1]
      logits=torch.where(logits<mini_val,torch.tensor(float("-inf")),logits)
    if temp>0:
      logits=logits/temp
      probs=torch.softmax(logits,dim=-1)
      idx_next=torch.multinomial(probs,num_samples=1)
    else:
      idx_next=torch.argmax(logits,dim=-1,keepdim=True)
    if idx_next==eos_id:
      break
    idx=torch.cat((idx,idx_next),dim=1)
  return idx


In [ ]:
torch.manual_seed(123)
token_ids=generate(
    model=model,
    idx=text_to_token("Every effort moves you",tokenizer),
    max_new_tokens=15,
    context_size=GPT_CONFIG_124M["context_length"],
    top_k=25,
    temp=1.6,
)
print(token_ids_to_text(token_ids,tokenizer))

Every effort moves youEveryiliaralso stabbed OrleansAllowsean 52anche crime winter unbeaten quoteembedreportprint earning


In [ ]:
model=GPTMOdel(GPT_CONFIG_124M)
torch.save(model.state_dict(),"model.pth")

In [ ]:
model=GPTMOdel(GPT_CONFIG_124M)
model.load_state_dict(torch.load("model.pth"))
model.eval()

GPTMOdel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(256, 768)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=False)
        (W_key): Linear(in_features=768, out_features=768, bias=False)
        (W_value): Linear(in_features=768, out_features=768, bias=False)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features

In [ ]:
optimizer=torch.optim.AdamW(model.parameters(),lr=0.0004,weight_decay=0.1)
torch.save({
    "model_state_dict":model.state_dict(),
    "optimizer_state_dict":optimizer.state_dict(),
    },
    "model_and_optimizer.pth"
)

In [ ]:
check=torch.load("model_and_optimizer.pth")
model=GPTMOdel(GPT_CONFIG_124M)
model.load_state_dict(check["model_state_dict"])
optimizer=torch.optim.AdamW(model.parameters(),lr=5e-4,weight_decay=0.1)

optimizer.load_state_dict(check["optimizer_state_dict"])
model.train()

GPTMOdel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(256, 768)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=False)
        (W_key): Linear(in_features=768, out_features=768, bias=False)
        (W_value): Linear(in_features=768, out_features=768, bias=False)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features

In [ ]:
pip install tensorflow>=2.15.0 tqdm>=4.66

In [ ]:
import tensorflow as tf
import tqdm

In [ ]:
from gpt3_download import download_and_load_gpt2

In [ ]:
settings,params=download_and_load_gpt2(model_size="124M",models_dir="gpt2")

/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'openaipublic.blob.core.windows.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
checkpoint: 100%|██████████| 77.0/77.0 [00:00<00:00, 164kiB/s]
/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'openaipublic.blob.core.windows.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
encoder.json: 100%|██████████| 1.04M/1.04M [00:00<00:00, 2.65MiB/s]
/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'openaipublic.blob.core.windows.net'. Adding certificate verificat

In [ ]:
print(settings)
print(params.keys())

{'n_vocab': 50257, 'n_ctx': 1024, 'n_embd': 768, 'n_head': 12, 'n_layer': 12}
dict_keys(['blocks', 'b', 'g', 'wpe', 'wte'])


In [ ]:
print(params["wte"])
print(params["wte"].shape)

[[-0.11010301 -0.03926672  0.03310751 ... -0.1363697   0.01506208
   0.04531523]
 [ 0.04034033 -0.04861503  0.04624869 ...  0.08605453  0.00253983
   0.04318958]
 [-0.12746179  0.04793796  0.18410145 ...  0.08991534 -0.12972379
  -0.08785918]
 ...
 [-0.04453601 -0.05483596  0.01225674 ...  0.10435229  0.09783269
  -0.06952604]
 [ 0.1860082   0.01665728  0.04611587 ... -0.09625227  0.07847701
  -0.02245961]
 [ 0.05135201 -0.02768905  0.0499369  ...  0.00704835  0.15519823
   0.12067825]]
(50257, 768)


In [ ]:
new_config=GPT_CONFIG_124M.copy()
new_config.update({"context_length":1024,"qkv_bias":True})
gpt=GPTMOdel(new_config)
gpt.eval()

GPTMOdel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=True)
        (W_key): Linear(in_features=768, out_features=768, bias=True)
        (W_value): Linear(in_features=768, out_features=768, bias=True)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=7

In [ ]:
class TransformerBlock(nn.Module):
  def __init__(self,cfg):
    super().__init__()
    self.att=MultiHeadAttention(
        d_in=cfg["emb_dim"],
        d_out=cfg["emb_dim"],
        context_length=cfg["context_length"],
        n_heads=cfg["n_heads"],
        dropout=cfg["drop_rate"],
        qkv_bias=cfg["qkv_bias"]
    )
    self.ff=FeedForward(cfg)
    self.norm1=LayerNorm(cfg["emb_dim"])
    self.norm2=LayerNorm(cfg["emb_dim"])
    self.drop_shortcut=nn.Dropout(cfg["drop_rate"])
  def forward(self,x):
    shortcut=x
    x=self.norm1(x)
    x=self.att(x)
    x=self.drop_shortcut(x)
    x=x+shortcut

    shortcut=x
    x=self.norm2(x)
    x=self.ff(x)
    x=self.drop_shortcut(x)
    x=x+shortcut
    return x

In [ ]:
def assign(left,right):
  if left.shape != right.shape:
    return ValueError(f"Shape MisMatch")
  return torch.nn.Parameter(torch.tensor(right))

In [ ]:
import numpy as np

def load_weights_into_gpt(gpt, params):
    gpt.pos_emb.weight = assign(gpt.pos_emb.weight, params['wpe'])
    gpt.tok_emb.weight = assign(gpt.tok_emb.weight, params['wte'])

    for b in range(len(params["blocks"])):
        q_w, k_w, v_w = np.split(
            (params["blocks"][b]["attn"]["c_attn"])["w"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.weight = assign(
            gpt.trf_blocks[b].att.W_query.weight, q_w.T)
        gpt.trf_blocks[b].att.W_key.weight = assign(
            gpt.trf_blocks[b].att.W_key.weight, k_w.T)
        gpt.trf_blocks[b].att.W_value.weight = assign(
            gpt.trf_blocks[b].att.W_value.weight, v_w.T)

        q_b, k_b, v_b = np.split(
            (params["blocks"][b]["attn"]["c_attn"])["b"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.bias = assign(
            gpt.trf_blocks[b].att.W_query.bias, q_b)
        gpt.trf_blocks[b].att.W_key.bias = assign(
            gpt.trf_blocks[b].att.W_key.bias, k_b)
        gpt.trf_blocks[b].att.W_value.bias = assign(
            gpt.trf_blocks[b].att.W_value.bias, v_b)

        gpt.trf_blocks[b].att.out_proj.weight = assign(
            gpt.trf_blocks[b].att.out_proj.weight,
            params["blocks"][b]["attn"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].att.out_proj.bias = assign(
            gpt.trf_blocks[b].att.out_proj.bias,
            params["blocks"][b]["attn"]["c_proj"]["b"])

        gpt.trf_blocks[b].ff.layers[0].weight = assign(
            gpt.trf_blocks[b].ff.layers[0].weight,
            params["blocks"][b]["mlp"]["c_fc"]["w"].T)
        gpt.trf_blocks[b].ff.layers[0].bias = assign(
            gpt.trf_blocks[b].ff.layers[0].bias,
            params["blocks"][b]["mlp"]["c_fc"]["b"])
        gpt.trf_blocks[b].ff.layers[2].weight = assign(
            gpt.trf_blocks[b].ff.layers[2].weight,
            params["blocks"][b]["mlp"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].ff.layers[2].bias = assign(
            gpt.trf_blocks[b].ff.layers[2].bias,
            params["blocks"][b]["mlp"]["c_proj"]["b"])

        gpt.trf_blocks[b].norm1.scale = assign(
            gpt.trf_blocks[b].norm1.scale,
            params["blocks"][b]["ln_1"]["g"])
        gpt.trf_blocks[b].norm1.shift = assign(
            gpt.trf_blocks[b].norm1.shift,
            params["blocks"][b]["ln_1"]["b"])
        gpt.trf_blocks[b].norm2.scale = assign(
            gpt.trf_blocks[b].norm2.scale,
            params["blocks"][b]["ln_2"]["g"])
        gpt.trf_blocks[b].norm2.shift = assign(
            gpt.trf_blocks[b].norm2.shift,
            params["blocks"][b]["ln_2"]["b"])

    gpt.final_norm.scale = assign(gpt.final_norm.scale, params["g"])
    gpt.final_norm.shift = assign(gpt.final_norm.shift, params["b"])
    gpt.out_head.weight = assign(gpt.out_head.weight, params["wte"])



In [ ]:
load_weights_into_gpt(gpt,params)
gpt.to(device);

In [ ]:
torch.manual_seed(123)
token_ids=generate(
    model=gpt,
    idx=text_to_token("Virat is a cricketer who",tokenizer).to(device),
    max_new_tokens=50,
    context_size=new_config["context_length"],
    top_k=30,
    temp=0.7
)
print(token_ids_to_text(token_ids,tokenizer))

Virat is a cricketer who has been playing for Indian teams for over a decade. The 24-year-old has played in two World Cup finals and the ODI series against Sri Lanka.

He's had a fantastic career at the club, scoring 23 runs in 29


In [ ]:
import pandas as pd

df = pd.read_csv(
    "SMSSpamCollection",
    sep="\t",
    header=None,
    names=["label", "message"]
)
df

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [ ]:
print(df["label"].value_counts())

label
ham     4825
spam     747
Name: count, dtype: int64


In [ ]:
def create_balanced(df):
  num_spam=df[df["label"]=="spam"].shape[0]
  ham_subset=df[df["label"]=="ham"].sample(num_spam,random_state=123)
  balanced_df=pd.concat([ham_subset,df[df["label"]=="spam"]])
  return balanced_df
balanced_df=create_balanced(df)
print(balanced_df["label"].value_counts())

label
ham     747
spam    747
Name: count, dtype: int64


In [ ]:
balanced_df["label"]=balanced_df["label"].map({"ham":0,"spam":1})

In [ ]:
def random_split(df,train_frac,validation_frac):
  df=df.sample(frac=1,random_state=123).reset_index(drop=True)
  train_end=int(train_frac*len(df))
  val_end=train_end+int(validation_frac*len(df))
  train_df=df[:train_end]
  val_df=df[train_end:val_end]
  test_df=df[val_end:]
  return train_df,val_df,test_df
train_df,val_df,test_df=random_split(balanced_df,0.7,0.1)

In [ ]:
len(train_df)

1045

In [ ]:
train_df.to_csv("train_df",index=None)
val_df.to_csv("val_df",index=None)
test_df.to_csv("test_df",index=None)
train_df

,label,message
0,0,Dude how do you like the buff wind.
1,0,Tessy..pls do me a favor. Pls convey my birthd...
2,1,Reminder: You have not downloaded the content ...
3,1,Got what it takes 2 take part in the WRC Rally...
4,1,"Shop till u Drop, IS IT YOU, either 10K, 5K, £..."
...,...,...
1040,1,4mths half price Orange line rental & latest c...
1041,1,Thanks for the Vote. Now sing along with the s...
1042,1,IMPORTANT INFORMATION 4 ORANGE USER 0796XXXXXX...
1043,1,Urgent! call 09066612661 from landline. Your c...


In [ ]:
import torch
from troch.utils.data import dataset
class SpamDataset(dataset):
  def __inti__(self,csv_file,tokenizer,max_legth=None,pad_token=50256):
    self.data=pd.read_csv(csv_file)
    self.encoded_texts=[
        tokenizer.encode(text) for text in self.data["Text"]
    ]


ModuleNotFoundError: No module named 'troch'